# ============================================================
# MINDMITRA - FEATURE ENGINEERING AND MODEL TRAINING
# Synthetic Dataset Version
# ============================================================


In [10]:
# ============================================================
# 1. IMPORT REQUIRED LIBRARIES
# ============================================================

import os
import pandas as pd
import numpy as np
import joblib

from textblob import TextBlob

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [3]:
df = pd.read_csv("../data(Synthetic)/demo_wellness_data.csv", encoding='utf-8')
df.head()

,time_stamp,created_at,date,week,user_id,name,age_group,gender,occupation,feeling_text,...,social_media_hours,socialmedia_hours>3,physically_active,exercise,mood_score,stress_score,energy_score,socializing,sociallyconnected_score,user_remark
0,2026-10-01,2026-10-01,2026-10-01,40,demo_001,Demo User 01,15 - 20,Female,Employee,I had difficulty concentrating,...,3.3,1,Yes,1,7,7,8,3,6,Want to reduce screen time
1,2026-10-01,2026-10-01,2026-10-01,40,demo_002,Demo User 02,15 - 20,Prefer not to say,Student,I had a calm and peaceful day,...,0.0,0,Yes,1,7,6,6,4,7,Need better sleep
2,2026-10-01,2026-10-01,2026-10-01,40,demo_003,Demo User 03,21 - 30,Female,Employee,I had difficulty concentrating,...,2.8,0,Yes,1,7,6,4,3,1,Feeling relaxed
3,2026-10-01,2026-10-01,2026-10-01,40,demo_004,Demo User 04,15 - 20,Prefer not to say,Student,I had a productive day,...,1.3,0,Yes,1,7,3,8,4,9,Feeling positive
4,2026-10-01,2026-10-01,2026-10-01,40,demo_005,Demo User 05,21 - 30,Male,Other,I had difficulty concentrating,...,3.6,1,Yes,1,5,7,6,1,8,Feeling relaxed


In [7]:
# Check dataset information
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   time_stamp               100 non-null    object 
 1   created_at               100 non-null    object 
 2   date                     100 non-null    object 
 3   week                     100 non-null    int64  
 4   user_id                  100 non-null    object 
 5   name                     100 non-null    object 
 6   age_group                100 non-null    object 
 7   gender                   100 non-null    object 
 8   occupation               100 non-null    object 
 9   feeling_text             100 non-null    object 
 10  positive_text            100 non-null    object 
 11  anxiety_text             100 non-null    object 
 12  sleep_hours              100 non-null    float64
 13  sleep_quality            100 non-null    object 
 14  screen_time              10

In [6]:
print(df.shape)
print(df.columns.tolist())

(100, 25)
['time_stamp', 'created_at', 'date', 'week', 'user_id', 'name', 'age_group', 'gender', 'occupation', 'feeling_text', 'positive_text', 'anxiety_text', 'sleep_hours', 'sleep_quality', 'screen_time', 'social_media_hours', 'socialmedia_hours>3', 'physically_active', 'exercise', 'mood_score', 'stress_score', 'energy_score', 'socializing', 'sociallyconnected_score', 'user_remark']


In [9]:
# Check missing values
print(df.isnull().sum())

time_stamp                 0
created_at                 0
date                       0
week                       0
user_id                    0
name                       0
age_group                  0
gender                     0
occupation                 0
feeling_text               0
positive_text              0
anxiety_text               0
sleep_hours                0
sleep_quality              0
screen_time                0
social_media_hours         0
socialmedia_hours>3        0
physically_active          0
exercise                   0
mood_score                 0
stress_score               0
energy_score               0
socializing                0
sociallyconnected_score    0
user_remark                0
dtype: int64


In [11]:
# Check unique values in categorical columns

categorical_cols = [
    'age_group',
    'gender',
    'occupation',
    'sleep_quality',
    'physically_active',
    'exercise',
    'socializing'
]

for col in categorical_cols:
    print(f"\n{col}:")

    print(df[col].unique())


age_group:
['15-20' '21-30' '31-40']

gender:
['Female' 'Prefer not to say' 'Male']

occupation:
['Employee' 'Student' 'Other']

sleep_quality:
['Good' 'Average' 'Poor']

physically_active:
['Yes' 'No']

exercise:
[1 0]

socializing:
[3 4 1 2 5]


In [12]:
# ============================================================
# 4. FEATURE ENGINEERING - CATEGORICAL ENCODING
# ============================================================

In [13]:
df['age_group'] = (
    df['age_group']
    .astype(str)
    .str.strip()
    .str.replace(" ", "")   # "21-30"
    .str.lower()
)


In [14]:
df['age_group_encoded'] = df['age_group'].map({
    '15-20': 2,
    '21-30': 3,
    '31-40': 4
})

In [15]:
# Gender Encoding

df['gender_encoded'] = df['gender'].map({
    'Female': 0,
    'Male': 1,
    'Prefer not to say': 2
})

In [16]:
# Sleep Quality Encoding

df['sleep_quality'] = df['sleep_quality'].map({
    'Poor': 1,
    'Average': 2,
    'Good': 3
})

In [17]:
# Physical Activity Encoding

df['physically_active_encoded'] = df['physically_active'].map({
    'Yes': 1,
    'No': 0
})


In [18]:
# Social Media Usage Encoding

# 1 = More than 3 hours
# 0 = 3 hours or less

df['socialmedia_hours>3_encoded'] = (
    df['social_media_hours'] > 3
).astype(int)

In [19]:
# Rename Social Connectivity Column

df = df.rename(columns={
    'sociallyconnected_score': 'sociallyconnected__score'
})


print("\nCategorical Encoding Completed!")


Categorical Encoding Completed!


In [20]:
# ============================================================
# 5. SENTIMENT ANALYSIS USING TEXTBLOB
# ============================================================

# TextBlob polarity:
# -1 = Negative
#  0 = Neutral
# +1 = Positive


def get_sentiment(text):

    if pd.isna(text):
        return 0

    return TextBlob(str(text)).sentiment.polarity

In [21]:
# Sentiment Analysis for Feeling Text

df['feeling_sentiment'] = (
    df['feeling_text'].apply(get_sentiment)
)

In [22]:
# Sentiment Analysis for Positive Text

df['positive_sentiment'] = (
    df['positive_text'].apply(get_sentiment)
)

In [23]:
# Sentiment Analysis for Anxiety Text

df['anxiety_sentiment'] = (
    df['anxiety_text'].apply(get_sentiment)
)

In [25]:
# Sentiment Analysis for User Remarks

df['remark_sentiment'] = (
    df['user_remark'].apply(get_sentiment)
)

print("\nSentiment Analysis Completed!")


Sentiment Analysis Completed!


In [27]:
#Sentiment Analysis for Text Columns

In [26]:
# ============================================================
# 6. CHECK FINAL ENGINEERED FEATURES
# ============================================================

features = [

    'sleep_hours',
    'sleep_quality',
    'screen_time',
    'socialmedia_hours>3_encoded',
    'physically_active_encoded',
    'exercise',
    'mood_score',
    'stress_score',
    'energy_score',
    'socializing',
    'sociallyconnected__score',
    'age_group_encoded',
    'gender_encoded',
    'feeling_sentiment',
    'positive_sentiment',
    'anxiety_sentiment',
    'remark_sentiment'

]


print("\nFinal Feature Columns:")

print(features)


Final Feature Columns:
['sleep_hours', 'sleep_quality', 'screen_time', 'socialmedia_hours>3_encoded', 'physically_active_encoded', 'exercise', 'mood_score', 'stress_score', 'energy_score', 'socializing', 'sociallyconnected__score', 'age_group_encoded', 'gender_encoded', 'feeling_sentiment', 'positive_sentiment', 'anxiety_sentiment', 'remark_sentiment']


In [27]:
# Check whether all required columns exist

missing_features = [
    col for col in features
    if col not in df.columns
]

if missing_features:
    raise ValueError(
        f"Missing required features: {missing_features}"
    )

In [28]:
# Convert model features to numeric values

for col in features:
    df[col] = pd.to_numeric(
        df[col],
        errors='coerce'
    )


# Check missing values in final features

print("\nMissing Values in Features:")

print(df[features].isnull().sum())


# Stop if any feature has missing values

if df[features].isnull().any().any():

    raise ValueError(
        "Some engineered features contain missing values. "
        "Please check the encoding and source data."
    )


Missing Values in Features:
sleep_hours                    0
sleep_quality                  0
screen_time                    0
socialmedia_hours>3_encoded    0
physically_active_encoded      0
exercise                       0
mood_score                     0
stress_score                   0
energy_score                   0
socializing                    0
sociallyconnected__score       0
age_group_encoded              0
gender_encoded                 0
feeling_sentiment              0
positive_sentiment             0
anxiety_sentiment              0
remark_sentiment               0
dtype: int64


In [29]:
# ============================================================
# 7. CREATE TARGET VARIABLE - MENTAL HEALTH RISK
# ============================================================

# Original target-label rules retained.
#
# 0 = Low Risk
# 1 = Medium Risk
# 2 = High Risk


def mental_health_label(row):

    if row['stress_score'] >= 7 and row['sleep_hours'] < 6:

        return 2

    elif row['stress_score'] >= 5:

        return 1

    else:

        return 0


df['mental_health_risk'] = df.apply(
    mental_health_label,
    axis=1
)


print("\nRisk Categories Created!")

print(
    df['mental_health_risk']
    .value_counts()
    .sort_index()
)


Risk Categories Created!
mental_health_risk
0    40
1    50
2    10
Name: count, dtype: int64


In [31]:
# ============================================================
# 8. SELECT INPUT FEATURES AND TARGET
# ============================================================

X = df[features]

y = df['mental_health_risk']


print("\nInput Features Shape:", X.shape)

print("Target Shape:", y.shape)


Input Features Shape: (100, 17)
Target Shape: (100,)


In [32]:
# ============================================================
# 9. SPLIT DATA INTO TRAINING AND TESTING SETS
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42

)


print("\nTraining Data Shape:", X_train.shape)

print("Testing Data Shape:", X_test.shape)


Training Data Shape: (80, 17)
Testing Data Shape: (20, 17)


In [33]:
# ============================================================
# 10. TRAIN RANDOM FOREST CLASSIFIER
# ============================================================

model = RandomForestClassifier(

    n_estimators=200,
    random_state=42

)


model.fit(X_train, y_train)


print("\nRandom Forest Model Training Completed!")


Random Forest Model Training Completed!


In [34]:
# ============================================================
# 11. MAKE PREDICTIONS
# ============================================================

y_pred = model.predict(X_test)

In [35]:
# ============================================================
# 12. MODEL EVALUATION
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)


print("\nModel Accuracy:")

print(accuracy)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2],
        target_names=[
            'Low Risk',
            'Medium Risk',
            'High Risk'
        ],
        zero_division=0
    )
)


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1, 2]
    )
)


Model Accuracy:
1.0

Classification Report:
              precision    recall  f1-score   support

    Low Risk       1.00      1.00      1.00         9
 Medium Risk       1.00      1.00      1.00         8
   High Risk       1.00      1.00      1.00         3

    accuracy                           1.00        20
   macro avg       1.00      1.00      1.00        20
weighted avg       1.00      1.00      1.00        20


Confusion Matrix:
[[9 0 0]
 [0 8 0]
 [0 0 3]]


In [ ]:
# NOTE:
# The model achieved 100% accuracy on the synthetic test dataset.
# This result is influenced by the target labels being generated using
# predefined rules based on stress_score and sleep_hours, which are
# also included among the model's input features.
# Therefore, the high accuracy indicates that the model learned these
# predefined patterns and should not be interpreted as proof of
# clinically reliable mental health risk prediction.
# Further validation using independently labelled real-world data is required.

In [36]:
# ============================================================
# 13. FEATURE IMPORTANCE
# ============================================================

feature_importance = pd.DataFrame({

    'Feature': features,

    'Importance': model.feature_importances_

})


feature_importance = feature_importance.sort_values(

    by='Importance',

    ascending=False

)


print("\nFeature Importance:")

display(feature_importance)


Feature Importance:


,Feature,Importance
7,stress_score,0.431264
0,sleep_hours,0.124464
2,screen_time,0.076353
6,mood_score,0.065393
1,sleep_quality,0.062391
8,energy_score,0.050812
10,sociallyconnected__score,0.032243
11,age_group_encoded,0.023906
9,socializing,0.022474
16,remark_sentiment,0.019881


In [ ]:
#Save Model for Integration with Streamlit 

In [37]:
# ============================================================
# 14. SAVE TRAINED MODEL
# ============================================================

# Save the new synthetic-data model separately.
# Do not overwrite the original model.

model_folder = "../model"

os.makedirs(
    model_folder,
    exist_ok=True
)


model_path = os.path.join(

    model_folder,

    "mental_health_demo_model.pkl"

)


joblib.dump(
    model,
    model_path
)


print("\nModel Saved Successfully!")

print("Model Location:", model_path)


Model Saved Successfully!
Model Location: ../model\mental_health_demo_model.pkl


In [38]:
# ============================================================
# 15. VERIFY SAVED MODEL
# ============================================================

loaded_model = joblib.load(model_path)


print("\nSaved Model Loaded Successfully!")


# Test prediction using the first test record

sample_prediction = loaded_model.predict(
    X_test.iloc[[0]]
)


risk_labels = {

    0: "Low Risk",

    1: "Medium Risk",

    2: "High Risk"

}


print(
    "\nSample Predicted Risk:",
    risk_labels[sample_prediction[0]]
)


print("\nFEATURE ENGINEERING AND MODEL TRAINING COMPLETED!")



Saved Model Loaded Successfully!

Sample Predicted Risk: Low Risk

FEATURE ENGINEERING AND MODEL TRAINING COMPLETED!
